In [1]:
from pyspark.sql.functions import * 

StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 3, Finished, Available, Finished, False)

In [2]:
df_fct = spark.read.table("lh_insurance_gold.dbo.gold_fct_claims")
df_pol = spark.read.table("lh_insurance_gold.dbo.gold_dim_policy")
df_cust = spark.read.table("lh_insurance_gold.dbo.gold_fct_claims")


StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 4, Finished, Available, Finished, False)

In [5]:
## Top 10 High-value customers by claim Amount
display(spark.sql("""
    SELECT 
        c.full_name as CustomerName,
        sum(claim_amount) as TotalClaimAmount
    FROM gold_dim_customer as c
    JOIN gold_fct_claims as fc 
    ON c.cust_id = fc.cust_id
    GROUP by CustomerName
    ORDER by TotalClaimAmount DESC
    LIMIT 10
""")
)

StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8654fca-76b5-4737-99d6-72b8e83e4929)

In [6]:
## Claim approval rate by gender 
display(
    spark.sql("""
    SELECT 
        c.gender,
        sum(fc.is_approved) as approved_claims,
        sum(fc.is_rejected) as rejected_claims,
        count(fc.claim_id) as total_claims,
        round((sum(fc.is_approved) / count(fc.claim_id)) * 100, 2) as approval_rate_pct
    FROM gold_dim_customer as c  
    JOIN gold_fct_claims as fc  
        ON c.cust_id = fc.cust_id
    GROUP BY 
        c.gender
""")
)

StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 821c42df-3b36-403e-9a80-364ebf328925)

In [7]:
## Claim to coverage ratio

display(
    spark.sql("""
    WITH cust_claims AS (
        SELECT 
            cust_id,
            sum(claim_amount) as total_claim_amount,
            count(claim_id) as total_claims
        FROM gold_fct_claims
        GROUP BY cust_id
    ),
    cust_policies AS (
        SELECT 
            cust_id,
            sum(coverage_amount) as total_coverage_amount,
            count(policy_id) as total_policies
        FROM gold_dim_policy
        GROUP BY cust_id
    )
    SELECT 
        c.cust_id,
        c.full_name,
        c.gender,
        coalesce(cp.total_coverage_amount, 0) as total_coverage_amount,
        coalesce(cc.total_claim_amount, 0) as total_claim_amount,
        coalesce(cc.total_claims, 0) as total_claims,
        round(
            coalesce(cc.total_claim_amount, 0) / nullif(cp.total_coverage_amount, 0), 
            4
        ) as claim_to_coverage_ratio,
        round(
            (coalesce(cc.total_claim_amount, 0) / nullif(cp.total_coverage_amount, 0)) * 100, 
            2
        ) as claim_to_coverage_pct
    FROM gold_dim_customer c
    LEFT JOIN cust_claims cc 
        ON c.cust_id = cc.cust_id
    LEFT JOIN cust_policies cp 
        ON c.cust_id = cp.cust_id
""")
)

StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 88933376-7879-41f0-b3b2-f8693d2244ba)

In [8]:
## claim by policy type 

display(
    spark.sql("""
    SELECT 
        p.policy_type,
        count(fc.claim_id) as total_claims,
        sum(fc.claim_amount) as total_claim_amount,
        round(avg(fc.claim_amount), 2) as avg_claim_amount,
        sum(fc.is_approved) as approved_claims,
        sum(fc.is_rejected) as rejected_claims,
        round((sum(fc.is_approved) / count(fc.claim_id)) * 100, 2) as approval_rate_pct
    FROM gold_dim_policy p
    JOIN gold_fct_claims fc
        ON p.policy_id = fc.policy_id
    GROUP BY 
        p.policy_type
    ORDER BY 
        total_claim_amount DESC
""")
)

StatementMeta(, 7e3e179a-2149-419c-b309-7a27ad79c008, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e31e7f95-46a8-4499-896c-86c6a84fdffa)

In [2]:
## median calculation for coverage amount and claim amount
# Customer-level median coverage

df_fct = spark.read.table("lh_insurance_gold.dbo.gold_fct_claims")
df_pol = spark.read.table("lh_insurance_gold.dbo.gold_dim_policy")
df_cust = spark.read.table("lh_insurance_gold.dbo.gold_fct_claims")


df_cust_cov = df_pol.groupBy("cust_id").sum("coverage_amount")
df_cust_claim = df_fct.groupBy("cust_id").sum("claim_amount")

print("Median Coverage (X):", df_cust_cov.approxQuantile("sum(coverage_amount)", [0.5], 0.01)[0])
print("Median Claim (Y):", df_cust_claim.approxQuantile("sum(claim_amount)", [0.5], 0.01)[0])

StatementMeta(, 229c52c5-cfa0-4a19-ba36-97fe3199557f, 4, Finished, Available, Finished, False)

Median Coverage (X): 79281.62
Median Claim (Y): 31563.79
